1. DOWNLOAD

In [2]:
!curl -O https://ftp.ncbi.nlm.nih.gov/pub/clinvar/vcf_GRCh37/clinvar.vcf.gz
!gunzip clinvar.vcf.gz
!curl -O https://ftp.ncbi.nlm.nih.gov/pub/clinvar/tab_delimited/variant_summary.txt.gz
!gunzip variant_summary.txt.gz
!curl -O https://ftp.ebi.ac.uk/pub/databases/Pfam/current_release/Pfam-A.hmm.gz
!gunzip Pfam-A.hmm.gz
!hmmpress Pfam-A.hmm
!pip install fair-esm torch numpy pandas scipy scikit-learn shap matplotlib requests -q

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  182M  100  182M    0     0  5993k      0  0:00:31  0:00:31 --:--:-- 5369kk      0  0:00:25  0:00:10  0:00:15 5784k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  418M  100  418M    0     0  5449k      0  0:01:18  0:01:18 --:--:-- 5048k 0  0:01:15  0:00:09  0:01:06 5246k0  0:01:15  0:00:11  0:01:04 5382k0  0:01:17  0:01:05  0:00:12 5813k00:06 5462k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  366M  100  366M    0     0  1499k      0  0:04:10  0:04:10 --:--:-- 1202k22  0:00:04  0:06:18  980k0  0:03:21  0:00:20  0:03:01 1384k:03:37  0:00:36  0:03:01 1524k1683k      0  0:03:42  0:00:47  0:02:55 1564k

In [1]:
import re, math, json, os, time, subprocess, pickle
import pandas as pd
import requests

2. PARSE VARIANTS

In [2]:
AA3TO1 = {
    'Ala':'A','Arg':'R','Asn':'N','Asp':'D','Cys':'C',
    'Gln':'Q','Glu':'E','Gly':'G','His':'H','Ile':'I',
    'Leu':'L','Lys':'K','Met':'M','Phe':'F','Pro':'P',
    'Ser':'S','Thr':'T','Trp':'W','Tyr':'Y','Val':'V'
}

def parse_protein_change(name):
    m = re.search(r'p\.([A-Za-z]{3})(\d+)([A-Za-z]{3})', str(name))
    if not m:
        return None, None, None
    return AA3TO1.get(m.group(1)), int(m.group(2)), AA3TO1.get(m.group(3))

vs = pd.read_csv('variant_summary.txt', sep='\t', low_memory=False,
                 usecols=['GeneSymbol','Name','ClinicalSignificance','Assembly','Type'])
vs = vs[(vs['Assembly']=='GRCh37') & (vs['Type']=='single nucleotide variant')]
vs[['wt','pos','mut']] = vs['Name'].apply(lambda x: pd.Series(parse_protein_change(x)))
vs = vs.dropna(subset=['wt','pos','mut'])
vs = vs[vs['mut'] != '*']
vs['pos'] = vs['pos'].astype(int)
vs['label'] = vs['ClinicalSignificance'].apply(
    lambda x: 1 if 'Pathogenic' in str(x) and 'Conflicting' not in str(x)
              else (0 if 'Benign' in str(x) and 'Conflicting' not in str(x) else None)
)
vs = vs.dropna(subset=['label'])
vs['label'] = vs['label'].astype(int)

top_genes   = vs['GeneSymbol'].value_counts().head(500).index.tolist()
clinvar_top = vs[vs['GeneSymbol'].isin(top_genes)].copy()

# Sample 5 per gene
clinvar_sample = clinvar_top.groupby('GeneSymbol').apply(
    lambda x: x.sample(min(len(x), 5), random_state=42)
).reset_index(drop=True)

print(f"clinvar_sample: {len(clinvar_sample)}")
clinvar_sample.to_csv('clinvar_sample.csv', index=False)

clinvar_sample: 2500


/var/folders/45/5q4rcg350xddpwn8646br6p00000gn/T/ipykernel_7433/2470832120.py:32: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  clinvar_sample = clinvar_top.groupby('GeneSymbol').apply(


3. PARSE Pfam

In [3]:
AA_ORDER = list("ACDEFGHIKLMNPQRSTVWY")

def parse_pfam(hmm_path):
    pfam = {}
    current = None
    in_hmm  = False
    with open(hmm_path) as f:
        for line in f:
            line = line.strip()
            if line.startswith('NAME'):
                current = line.split()[1]
                pfam[current] = {}
                in_hmm = False
            elif line.startswith('HMM '):
                in_hmm = True
                next(f); next(f)
            elif in_hmm and line and line[0].isdigit():
                parts = line.split()
                try:
                    pos = int(parts[0])
                except ValueError:
                    continue
                pfam[current][pos] = {
                    aa: math.exp(-float(v)) if v != '*' else 0.0
                    for aa, v in zip(AA_ORDER, parts[1:21])
                }
    print(f"Pfam: {len(pfam)} domains")
    return pfam

pfam = parse_pfam('Pfam-A.hmm')
pickle.dump(pfam, open('pfam.pkl', 'wb'))

Pfam: 27481 domains


4. UniProt IDs and Sequences

In [4]:
def get_uniprot_id(gene):
    url = f"https://rest.uniprot.org/uniprotkb/search?query={gene}+AND+organism_id:9606+AND+reviewed:true&format=json&size=1"
    try:
        r = requests.get(url, timeout=10)
        d = r.json()
        if d['results']:
            return d['results'][0]['primaryAccession']
    except:
        pass
    return None

def get_sequence(uid):
    try:
        r = requests.get(f"https://rest.uniprot.org/uniprotkb/{uid}.fasta", timeout=10)
        lines = r.text.strip().split('\n')
        return ''.join(lines[1:])
    except:
        return None

uid_mapping = {}
sequences   = {}

for i, gene in enumerate(clinvar_sample['GeneSymbol'].unique()):
    if i % 20 == 0:
        print(f"[{i}] {gene}")
        json.dump(uid_mapping, open('uid_mapping.json', 'w'))
        json.dump(sequences,   open('sequences.json',   'w'))
    uid = get_uniprot_id(gene)
    if not uid:
        continue
    seq = get_sequence(uid)
    if not seq:
        continue
    uid_mapping[gene] = uid
    sequences[gene]   = seq
    time.sleep(0.3)

json.dump(uid_mapping, open('uid_mapping.json', 'w'))
json.dump(sequences,   open('sequences.json',   'w'))
print(f"Done: {len(sequences)} sequences")

[0] A2ML1
[20] AGXT
[40] ARID1A
[60] AVPR2
[80] CAMK2B
[100] COL11A2
[120] CRB1
[140] DNAH5
[160] EPG5
[180] FGFR3
[200] GABRB2
[220] GNE
[240] IDUA
[260] KCNH5
[280] L1CAM
[300] MMUT
[320] NALCN
[340] OBSCN
[360] PHEX
[380] PSEN1
[400] RHOBTB2
[420] SERPINC1
[440] SOS1
[460] TGM1
[480] TUBB3
Done: 267 sequences


5. HMMER

In [5]:
def run_hmmer(gene, seq):
    fasta  = f'/tmp/{gene}.fasta'
    result = f'/tmp/{gene}_result.txt'
    with open(fasta, 'w') as f:
        f.write(f">{gene}\n{seq}\n")
    subprocess.run(['hmmscan','--domtblout',result,'--noali','Pfam-A.hmm',fasta],
                   capture_output=True)
    return result

def parse_hmmer(path):
    domains = []
    try:
        with open(path) as f:
            for line in f:
                if line.startswith('#') or not line.strip():
                    continue
                parts = line.split()
                if len(parts) >= 19 and float(parts[12]) < 0.001:
                    domains.append((parts[0], int(parts[17]), int(parts[18])))
    except:
        pass
    return domains

gene_mapping = {}
for i, (gene, seq) in enumerate(sequences.items()):
    if i % 20 == 0:
        print(f"[{i}] {gene}")
        json.dump(gene_mapping, open('gene_mapping.json', 'w'))
    gene_mapping[gene] = parse_hmmer(run_hmmer(gene, seq))

json.dump(gene_mapping, open('gene_mapping.json', 'w'))
print(f"Done: {len(gene_mapping)} genes")

[0] A2ML1
[20] AGXT
[40] ARID1A
[60] AVPR2
[80] CAMK2B
[100] COL11A2
[120] CRB1
[140] DNAH5
[160] EPG5
[180] FGFR3
[200] GABRB2
[220] GNE
[240] IFIH1
[260] KCNJ11
Done: 267 genes


6. Parse FATHMM Output

In [6]:
# Submit fathmm_input.txt to http://fathmm.biocompute.org.uk/inherited.html first

# Generate input file
lines = []
for _, row in clinvar_sample.iterrows():
    uid = uid_mapping.get(row['GeneSymbol'])
    if uid:
        lines.append(f"{uid} {row['wt']}{int(row['pos'])}{row['mut']}")

with open('fathmm_input.txt', 'w') as f:
    f.write('\n'.join(set(lines)))

print(f"Submit fathmm_input.txt to FATHMM web server")
print(f"Save result as fathmm_output.rtf")

Submit fathmm_input.txt to FATHMM web server
Save result as fathmm_output.rtf
